In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
from upsetplot import UpSet, generate_counts, plot, from_contents
from sklearn.datasets import load_diabetes

In [ ]:
#load input data
data = pd.read_csv("AllMetaproteins.csv", sep=";", index_col=0)
data.head(10)

In [ ]:
### specify data type to enable test if metaprotein abundance is >0
for row in data.index:
    if row.startswith("Meta-Protein"):
        data.loc[row]=data.loc[row].astype(float)

# Sort into Discovery, Validation und Disease Specificity Datasets
# Access row 4 (UseCase)
row = data.loc["UseCase"]

discovery_columns = row[row == 'BiomarkerDiscovery'].index
validation_columns = row[row == 'BiomarkerValidation'].index
specificity_columns = row[row == 'DiseaseSpecificity'].index

df_discovery = data[discovery_columns]
df_validation = data[validation_columns]
df_specificity = data[specificity_columns]

UseCaseSpecData={"discovery":df_discovery,"validation":df_validation,"specificity":df_specificity}
print(len(df_discovery.columns))


In [ ]:
# data transformation
datasets= {"Lehmann":[], "Henry":[], "Thuy-Boun":[], "Lloyd-Price":[]}

# initiate categories as input for upset plot

# select for each categorie (study) metaproteins that were identified in this study
for dataset in datasets.keys():
    # use only subset of samples according to the category and without metadata
    row = df_discovery.loc["study"]
    category_columns = row[row == dataset].index
    data = df_discovery[category_columns]
    data = data.drop(["study","condition","disease","disease_state","UseCase"],axis=0)
    
    # test wether metaprotein has cummulated abundace >0, if true, append to list
    for metaprotein in data.index:
        if data.loc[metaprotein].sum() > 0:
            datasets[dataset].append(metaprotein)
        

upsetData = from_contents(
    datasets
)
upsetData

In [ ]:
### plot data and highlight the insection of all 4

# "Lehmann": "#4472c4",
# "Thuy-Boun": "#33a02c",
# "Henry": "#ff0000",
# "Lloyd-Price": "#cab2d6",

upset = UpSet(upsetData, show_counts="{:,}")
print(type(upset))

# color the numbers of total metaprotein in distinct colours
upset.style_categories("Lehmann", bar_facecolor="#4472c4", bar_edgecolor="black", bar_linewidth=2)
upset.style_categories("Henry", bar_facecolor="#33a02c", bar_edgecolor="black", bar_linewidth=2)
upset.style_categories("Lloyd-Price", bar_facecolor="#cab2d6", bar_edgecolor="black", bar_linewidth=2)
upset.style_categories("Thuy-Boun", bar_facecolor="#ff0000", bar_edgecolor="black", bar_linewidth=2)


# color the sets off unique metaproteins in distinct colours
upset.style_subsets(present="Lehmann", absent=["Lloyd-Price","Henry","Thuy-Boun"], facecolor="#4472c4", linewidth=2)
upset.style_subsets(present="Henry", absent=["Lloyd-Price","Lehmann","Thuy-Boun"], facecolor="#33a02c", linewidth=2)
upset.style_subsets(present="Lloyd-Price", absent=["Lehmann","Henry","Thuy-Boun"], facecolor="#cab2d6", linewidth=2)
upset.style_subsets(present="Thuy-Boun", absent=["Lloyd-Price","Henry","Lehmann"], facecolor="#ff0000", linewidth=2)

# highlight shared section
upset.style_subsets(present=["Lloyd-Price","Henry","Thuy-Boun", "Lehmann"], edgecolor="red", linewidth=2, label="Identified in all studies")

upset.plot()
#upset.savefig(folderPath+"UpsetPlot", dpi=300, bbox_inches='tight')  # dpi controls the resolution
